# Purpose of this notebook
# Input: a folder containing one CSV file per pose
# Output: clustered poses and a visualization of the keypoints

### Preprocessing step

We apply preprocessing to the CSV files containing the pose detections:

- For each keypoint, subtract the `x` and `y` coordinates of the bounding box.  
  (This also applies to the bounding box coordinates themselves.)

- Then, normalize the pixel values to fit within a [0, 1] range,  
  where (1,1) corresponds to the **bottom-right** corner of the bounding box.

- The calculation is as follows:
  - `x_new = x / bottom_right_x`
  - `y_new = y / top_left_y`




In [ ]:
import os
import pandas as pd

def preprocess_csv(file_path, output_folder):
    # load the CSV file
    data = pd.read_csv(file_path, delimiter=',')
    
    # apply transformations to each row
    for index, row in data.iterrows():
        bbox_xmin, bbox_ymin, bbox_xmax, bbox_ymax = row[['bbox_xmin', 'bbox_ymin', 'bbox_xmax', 'bbox_ymax']]
        for col in data.columns:
            if '_x' in col:
                if row[col] != 0:
                    data.at[index, col] = (row[col] - bbox_xmin) / (bbox_xmax - bbox_xmin)
            elif '_y' in col:
                if row[col] != 0:
                    data.at[index, col] = (row[col] - bbox_ymin) / (bbox_ymax - bbox_ymin)
        
        # normalize bounding box coordinates
        data.at[index, 'bbox_xmin'] = 0
        data.at[index, 'bbox_ymin'] = 0
        data.at[index, 'bbox_xmax'] = 1
        data.at[index, 'bbox_ymax'] = 1

    # determine the output file path
    output_file_name = os.path.splitext(os.path.basename(file_path))[0] + '-changement_de_repere.csv'
    output_file_path = os.path.join(output_folder, output_file_name)
    
    # save the preprocessed file
    data.to_csv(output_file_path, index=False)

    return output_file_path

def preprocess_all_files(input_folder, output_folder):
    preprocessed_files = []
    for filename in os.listdir(input_folder):
        if filename.endswith('.csv'):
            file_path = os.path.join(input_folder, filename)
            preprocessed_file = preprocess_csv(file_path, output_folder)
            preprocessed_files.append(preprocessed_file)
    return preprocessed_files

# path to the folder containing the CSV files
input_folder_path = ''

# path to the output folder where preprocessed files will be saved
output_folder_path = ''

# check if the output folder exists, create it if not
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

# preprocess all CSV files
preprocessed_files = preprocess_all_files(input_folder_path, output_folder_path)

print("Preprocessing complete. The preprocessed files are:")
for file in preprocessed_files:
    print(file)


This script applies K-means clustering to all pose keypoint CSV files in a given folder.
It normalizes the data, groups similar poses, and saves the clustered data into separate CSV files.


In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# function to load all CSV files in a folder
def load_csv_files(folder_path):
    all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
    dataframes = [pd.read_csv(file, delimiter=',') for file in all_files]  # Change delimiter to ','
    combined_df = pd.concat(dataframes, ignore_index=True)
    return combined_df

# function to apply K-means and save the results
def kmeans_clustering(folder_path, n_clusters):
    data = load_csv_files(folder_path)

    # display available columns for diagnostics
    print("Available columns: ", data.columns)
    
    # check if 'nom_image' is in the columns before trying to drop it
    if 'nom_image' in data.columns:
        features = data.drop(columns=['nom_image'])
    else:
        print("'nom_image' not found in columns. Using all columns.")
        features = data
    
    # convert data to numeric format if necessary
    features = features.apply(pd.to_numeric, errors='coerce')
    
    # drop rows containing NaN values after conversion
    features = features.dropna()

    # normalize the features
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    # apply K-means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    data['cluster'] = kmeans.fit_predict(scaled_features)

    # create a folder for the clusters
    output_folder = os.path.join(folder_path, 'clusters')
    os.makedirs(output_folder, exist_ok=True)

    # save data per cluster
    for cluster in range(n_clusters):
        cluster_data = data[data['cluster'] == cluster]
        output_file = os.path.join(output_folder, f'cluster_{cluster}.csv')
        cluster_data.to_csv(output_file, index=False)

    print(f'Clustering complete. Output files are saved in {output_folder}.')

# path to the folder containing the CSV files
folder_path = ''

# apply K-means clustering
kmeans_clustering(folder_path, n_clusters=50)


Draw the bouding box and the body for one cluster 

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_cluster_data(file_path):
    # load data from the specified file using ',' as the delimiter
    data = pd.read_csv(file_path, delimiter=',')
    print(f"Loaded {file_path} with columns: {data.columns.tolist()}")
    return data

def calculate_means(cluster_data):
    # check for expected columns
    expected_bbox_columns = ['bbox_xmin', 'bbox_ymin', 'bbox_xmax', 'bbox_ymax']
    if not all(col in cluster_data.columns for col in expected_bbox_columns):
        raise KeyError(f"Expected bounding box columns not found in data: {expected_bbox_columns}")
    
    keypoints_columns = [col for col in cluster_data.columns if col not in ['nom_image', 'bbox_xmin', 'bbox_ymin', 'bbox_xmax', 'bbox_ymax', 'cluster']]

    bbox_means = cluster_data[expected_bbox_columns].mean().values
    keypoints_means = cluster_data[keypoints_columns].mean().values

    return bbox_means, keypoints_means

def plot_keypoints(bbox_means, keypoints_means, output_path):
    # prepare keypoint data
    keypoints_x = keypoints_means[::2]  # all x coordinates
    keypoints_y = keypoints_means[1::2]  # all y coordinates

    # create a blank figure
    fig, ax = plt.subplots()
    ax.set_xlim(0, 2)
    ax.set_ylim(0, 1)
    ax.invert_yaxis()  # invert the y-axis to match image orientation

    # draw the bounding box
    bbox_xmin, bbox_ymin, bbox_xmax, bbox_ymax = bbox_means
    rect = plt.Rectangle((bbox_xmin, bbox_ymin), bbox_xmax - bbox_xmin, bbox_ymax - bbox_ymin,
                         linewidth=1, edgecolor='r', facecolor='none')
    ax.add_patch(rect)

    # draw the keypoints
    ax.scatter(keypoints_x, keypoints_y, c='b', marker='o')

    # save the figure
    plt.savefig(output_path)
    plt.close(fig)

def visualize_all_clusters(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            cluster_num = os.path.splitext(filename)[0].split('_')[-1]
            cluster_data = load_cluster_data(os.path.join(folder_path, filename))
            try:
                bbox_means, keypoints_means = calculate_means(cluster_data)
                output_path = os.path.join(folder_path, f'cluster_{cluster_num}_mean.png')
                plot_keypoints(bbox_means, keypoints_means, output_path)
                print(f"Saved visualization for cluster {cluster_num} at {output_path}")
            except KeyError as e:
                print(f"Error processing {filename}: {e}")

# path to the folder containing cluster files
folder_path = ''

visualize_all_clusters(folder_path)
